# 03a — Video Mode 1: First/Last(/Middle)-Frame → Video (LTX-Video 0.9.8-13B-distilled)

Replaces the ComfyUI LTX-Video FLF2V workflow (`03_video_mode1_ltx_flf2v.json`). Pure diffusers:
`LTXConditionPipeline` + `LTXVideoCondition` — no custom nodes, no server.

**What it does:** takes a start still (required), an end still (optional → plain I2V), and optional
middle keyframes, then generates the video between them. This is the workhorse for Mode 3 (agentic)
scene interpolation.

**Why this model (spec 03 default):** Apache-2.0, guidance/timestep-distilled (8 steps + 5-step
upscale refine pass ≈ a 13B-quality clip in a couple of minutes on A100), supports *multiple*
keyframes in one pass (ComfyUI's `LTXVAddGuide` did the same thing).

**Inputs** (paths in Drive, from 02a stills or uploaded here):
- `start frame` — required still of the character
- `end frame` — optional target still
- `middle frames` — optional `{path, fraction}` list, fraction ∈ (0,1) of clip length

**VRAM:** 13B distilled ≈ 26 GB bf16 + VAE decode. Strategy auto-selected; 40 GB A100 uses group
offloading, 80 GB runs resident. T4/free tier → see the commented 2B / GGUF variants in §8.

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
NUM_FRAMES     = 121     # 8k+1 frames; 121 ≈ 5 s @ 24 fps
FPS            = 24
WIDTH, HEIGHT  = 832, 480   # landscape 16:9 LTX default; 480x832 portrait also fine
LORA_STRENGTH  = 1.4     # validated on real gens (01c); 02a uses the same
UPSCALE_2X     = True     # latent 2× upscale + 5-step refine pass (docs recipe)
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
VID_OUT    = f'{DRIVE_BASE}/outputs/videos/{CHARACTER_NAME}/mode1_ltx'
os.makedirs(VID_OUT, exist_ok=True)
# HF cache on LOCAL /content (fast + atomic-safe) for small extras (the upscaler). The big LTX-13B is downloaded straight to /content/ltx_13b in §3 (no Drive mirror — re-downloads each session).
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# Where your keyframes live (02a output dirs) — or set absolute paths below in §6
STILLS_ROOT = f'{DRIVE_BASE}/outputs/images/{CHARACTER_NAME}'
print(f'Video out : {VID_OUT}')
print(f'Stills root: {STILLS_ROOT}')

## 2. HuggingFace login + install (uv)
LTX-Video is Apache-2.0 — no gated license, token is only needed if you use a private cache. We still
pass the Colab Secrets token for consistency.

In [ ]:
import os
hf_token = ''
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
os.environ.setdefault('HF_TOKEN', hf_token)

!pip install -q uv
# ── torchao: remove BEFORE any diffusers/peft import in this kernel ────────
# Colab ships torchao 0.10.0 (incompatible with the peft diffusers pulls).
# diffusers>=0.40 bakes is_torchao_available() into a module-level flag at
# import time — if diffusers is imported while torchao is still present and we
# uninstall it afterwards, the flag stays True and every pipeline import
# crashes with "No module named 'torchao'". Uninstalling first means a fresh
# import caches False. (After a runtime restart the base image brings torchao
# back — the load cell's guard detects that and says to re-run this cell.)
!uv pip uninstall --system -y torchao 2>/dev/null || true
# diffusers must be recent enough for LTX 0.9.8 + latent upsample pipeline
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.35.0" transformers accelerate safetensors huggingface_hub

import torch, diffusers
print('torch', torch.__version__, '| diffusers', diffusers.__version__)
import importlib.util
print('torchao importable (want False):', importlib.util.find_spec('torchao') is not None)
# ── CUDA-torch guard ────────────────────────────────────────────────────────
# uv can silently re-resolve deps and swap Colab's CUDA torch for the PyPI CPU
# wheel ("Torch not compiled with CUDA enabled" at load time). Detect + repair
# here, BEFORE we commit to a multi-GB model download.
import torch as _t
print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
if not _t.cuda.is_available():
    import subprocess
    ver = _t.__version__.split('+')[0]
    print(f'CPU-only torch {ver} detected (uv swapped it). Reinstalling the CUDA build from cu124...')
    subprocess.run(f'uv pip install --system "torch=={ver}" torchvision '
                   '--index-url https://download.pytorch.org/whl/cu124',
                   shell=True, check=True)
    raise RuntimeError(
        'CUDA torch reinstalled on disk. Now: Runtime > Restart runtime, then re-run '
        'this install cell + the model-load cell. (The running kernel still holds the '
        'old CPU torch in memory, so the restart is required — do not skip it.)')


## 3. Load LTX-Video 0.9.8-13B-distilled (+ optional 2× latent upscaler)
First run downloads ~27 GB to the Drive HF cache. Log goes to a file (01c lesson: never an unread
PIPE).

In [ ]:
import torch, os, logging, importlib.util

# ── torchao stale-flag self-heal ──────────────────────────────────────────
# diffusers>=0.40 caches is_torchao_available() into a module-level bool at
# import time. Colab ships torchao 0.10.0 (incompatible); the install cell
# uninstalls it, but if diffusers was ALREADY imported this kernel (while
# torchao was still present) the flag stays True and every pipeline import
# crashes with "No module named 'torchao'". Reset the cached flag in-process
# when the module is really gone — no runtime restart needed.
from diffusers.utils import import_utils as _iu
if _iu._torchao_available and importlib.util.find_spec('torchao') is None:
    _iu._torchao_available = False
    _iu._torchao_version = 'N/A'
    print('torchao flag reset (cached True but module removed) — continuing.')

from diffusers import LTXConditionPipeline, LTXLatentUpsamplePipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.pipelines.ltx.modeling_latent_upsampler import LTXLatentUpsamplerModel

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}  ~{vram_gb:.0f} GB')

LOG = '/content/ltx_load.log'
logging.basicConfig(filename=LOG, level=logging.INFO)

# ── LTX-Video 0.9.8-13B on fast LOCAL /content disk ───────────────────────
# Downloaded straight to /content (local SSD, atomic-safe). NO Drive mirror:
# /content is wiped on a runtime reset, so we simply re-download here each
# session. snapshot_download is resumable/idempotent — it skips files already
# present (so a re-run in the same session is a no-op) and only fetches the
# gap, and it writes whole files atomically, so no partial/corrupt leftovers.
# (~27 GB, a few minutes on the A100 egress.)
LTX_REPO = 'Lightricks/LTX-Video-0.9.8-13B-distilled'
LTX_LOCAL = '/content/ltx_13b'
os.makedirs(LTX_LOCAL, exist_ok=True)

def _hf_token():
    t = os.environ.get('HF_TOKEN', '')
    if t:
        return t
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN') or ''
    except Exception:
        return ''

from huggingface_hub import snapshot_download
print(f'Ensuring {LTX_REPO} on local /content (downloads only what is missing)...')
snapshot_download(LTX_REPO, local_dir=LTX_LOCAL, max_workers=8, token=_hf_token() or None)

LTX_SRC = LTX_LOCAL
print(f'LTX source: {LTX_SRC}\n')

# ── Load / free helpers ────────────────────────────────────────────────────
# §5 swaps in FLUX to generate the keyframes (FLUX-resident ~34 GB does not
# share the A100 with LTX-resident ~30 GB). free_ltx() drops the pipeline, §5
# then calls load_ltx() again.

def free_ltx():
    # Drop the LTX pipeline from VRAM. Null-safe: works even if no pipeline
    # was loaded.
    #
    # Also sweep the *exception* references (sys.last_* and IPython's
    # _last_traceback). A failed `pipeline.to('cuda')` (CUDA OOM) leaves the
    # half-built ~35 GB pipeline pinned in the traceback's frame locals even
    # after `pipeline = None`, so a naive gc/empty_cache leaves a ghost in
    # VRAM and the next load OOMs on top of it. Clearing the tracebacks is
    # what actually releases it.
    global pipeline, pipe_upsample
    had = globals().get('pipeline') is not None
    pipeline = None
    pipe_upsample = None
    import gc, sys
    sys.last_type = sys.last_value = sys.last_traceback = None
    _ip = globals().get('get_ipython')
    if _ip is not None:
        try:
            _ip._last_traceback = None
        except Exception:
            pass
    gc.collect(); gc.collect()
    torch.cuda.empty_cache()
    if had:
        print(f'LTX freed — VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB')
    else:
        print('free_ltx: no LTX pipeline was loaded; swept stray refs. '
              f'VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

def load_ltx():
    # Free any pipeline already resident (and sweep OOM-ghost refs) so
    # re-running this cell never co-allocates two ~35 GB LTX pipelines
    # (CUDA OOM on 80 GB).
    free_ltx()
    pipeline = LTXConditionPipeline.from_pretrained(LTX_SRC, torch_dtype=torch.bfloat16)
    pipeline.vae.enable_tiling()
    if vram_gb >= 60:
        pipeline.to('cuda')
        print('Strategy: resident (>=60GB)')
    elif vram_gb >= 30:
        from diffusers.hooks import apply_group_offloading
        onload, offload = torch.device('cuda'), torch.device('cpu')
        pipeline.transformer.enable_group_offload(onload_device=onload, offload_device=offload,
                                                  offload_type='leaf_level', use_stream=True)
        apply_group_offloading(pipeline.text_encoder, onload_device=onload, offload_device=offload,
                               offload_type='block_level', num_blocks_per_group=2)
        apply_group_offloading(pipeline.vae, onload_device=onload, offload_device=offload,
                               offload_type='leaf_level')
        print('Strategy: group offloading (30-60GB)')
    else:
        pipeline.enable_model_cpu_offload()
        print('Strategy: CPU offload (<30GB) — expect slower; consider 2B variant (§8)')

    # Optional 2x latent upscaler (docs recipe: generate at 2/3 res, upscale,
    # 5-step refine). Small model -> local HF cache, never the FUSE cache.
    pipe_upsample = None
    if UPSCALE_2X:
        try:
            upsampler = LTXLatentUpsamplerModel.from_pretrained(
                'a-r-r-o-w/LTX-0.9.8-Latent-Upsampler', torch_dtype=torch.bfloat16)
            # .to('cuda') — a dtype-only .to(torch.bfloat16) leaves the
            # upsampler on CPU and the refine pass crashes with
            # "Input type (CUDA) and weight type (CPU) should be the same".
            pipe_upsample = LTXLatentUpsamplePipeline(vae=pipeline.vae,
                                                      latent_upsampler=upsampler).to('cuda')
            print('✅ Latent upscaler loaded (on cuda).')
        except Exception as e:
            print('Upscaler failed to load, will run single-pass:', str(e)[:150])
    print('✅ LTX pipeline ready.  Log →', LOG)
    return pipeline, pipe_upsample

pipeline, pipe_upsample = load_ltx()

## 4. The `interpolate()` helper (multi-keyframe)
Distilled-model settings straight from the official docs: `guidance_scale=1.0`,
`guidance_rescale=0.7`, custom timesteps, VAE dims rounded to multiples of 32.

Keyframes are `LTXVideoCondition(image=..., frame_index=N)` where `frame_index` is the position in
the final clip (0 = first, `NUM_FRAMES-1` = last, middle = `int(fraction * (NUM_FRAMES-1))`).
**Tip (from the docs):** use *similar* images for best results — big subject/lighting divergence
between keyframes causes abrupt transitions.

In [ ]:
import torch, time
from pathlib import Path
from PIL import Image
from diffusers.utils import export_to_video

def _round32(x):
    return int(x) // 32 * 32

def interpolate(start_frame, prompt, end_frame=None, middle_frames=None,
                width=WIDTH, height=HEIGHT, num_frames=NUM_FRAMES,
                seed=0, upscale=UPSCALE_2X, tag=''):
    """
    Generate a clip between keyframes.
      start_frame  : PIL image or path (required)
      end_frame    : PIL image or path (optional; omit for plain I2V)
      middle_frames: list of (path|PIL, fraction) e.g. [('mid.png', 0.5)]
      prompt       : describe MOTION/action; identity comes from the frames themselves.
    Returns path to the saved .mp4.
    """
    def load(p):
        return p if isinstance(p, Image.Image) else Image.open(p).convert('RGB')

    conditions = [LTXVideoCondition(image=load(start_frame), frame_index=0)]
    for path, frac in (middle_frames or []):
        conditions.append(LTXVideoCondition(image=load(path),
                                            frame_index=int(frac * (num_frames - 1))))
    if end_frame is not None:
        conditions.append(LTXVideoCondition(image=load(end_frame), frame_index=num_frames - 1))

    neg = 'worst quality, inconsistent motion, blurry, jittery, distorted'
    generator = torch.Generator().manual_seed(seed)

    h, w = _round32(height), _round32(width)

    if upscale and pipe_upsample is not None:
        # docs recipe: 2/3 res → 2x latent upscale → 5-step refine → resize to target
        dh, dw = _round32(h * 2 / 3), _round32(w * 2 / 3)
        latents = pipeline(
            conditions=conditions, prompt=prompt, negative_prompt=neg,
            width=dw, height=dh, num_frames=num_frames,
            timesteps=[1000, 993, 987, 981, 975, 909, 725, 0.03],
            decode_timestep=0.05, decode_noise_scale=0.025, image_cond_noise_scale=0.0,
            guidance_scale=1.0, guidance_rescale=0.7,
            generator=generator, output_type='latent',
        ).frames
        uh, uw = dh * 2, dw * 2
        # Device self-heal: the upsampler must live where the latents do
        # (a dtype-only .to() at load time can leave it on CPU).
        if next(pipe_upsample.latent_upsampler.parameters()).device != latents.device:
            pipe_upsample.latent_upsampler.to(latents.device)
        upscaled = pipe_upsample(latents=latents, adain_factor=1.0,
                                 tone_map_compression_ratio=0.6,
                                 output_type='latent').frames
        frames = pipeline(
            conditions=conditions, prompt=prompt, negative_prompt=neg,
            width=uw, height=uh, num_frames=num_frames,
            denoise_strength=0.999, timesteps=[1000, 909, 725, 421, 0],
            latents=upscaled, decode_timestep=0.05, decode_noise_scale=0.025,
            image_cond_noise_scale=0.0, guidance_scale=1.0, guidance_rescale=0.7,
            generator=generator, output_type='pil',
        ).frames[0]
        frames = [f.resize((w, h)) for f in frames]
    else:
        frames = pipeline(
            conditions=conditions, prompt=prompt, negative_prompt=neg,
            width=w, height=h, num_frames=num_frames,
            timesteps=[1000, 993, 987, 981, 975, 909, 725, 0.03],
            decode_timestep=0.05, decode_noise_scale=0.025, image_cond_noise_scale=0.0,
            guidance_scale=1.0, guidance_rescale=0.7,
            generator=generator, output_type='pil',
        ).frames[0]

    ts = time.strftime('%Y%m%d_%H%M%S')
    out = Path(VID_OUT) / f'{ts}_{tag}.mp4' if tag else Path(VID_OUT) / f'{ts}.mp4'
    export_to_video(frames, str(out), fps=FPS)
    print(f'✅ clip → {out}  ({num_frames} frames @ {FPS} fps, {w}x{h})')
    return str(out)

print('interpolate() ready.')

## 5. Generate keyframes (FLUX + character LoRA)
Both keyframes come from the **same scene** with a small, clear pose/expression change (a head
turn + smile) — the LTX docs tip: *similar* keyframes interpolate best, and a small demonstrable
delta lets us see if the first→last transition actually animates. Keyframes are generated at the
video resolution (832×480) so LTX doesn't resample them.

This cell **parks LTX** (weights stay on /content), loads FLUX.1-dev from the local staged copy
(01c/02a layout) + the character LoRA at `LORA_STRENGTH`, generates both frames, then **restores
LTX** — so §6 runs immediately after. (FLUX-resident ~34 GB and LTX-resident ~30 GB don't share
the A100, hence the park/restore dance.)


In [ ]:
# §5 — Generate fresh start/end keyframes with FLUX + character LoRA,
# then park FLUX and hand control back to LTX so §6 runs immediately after.
#
# Both keyframes describe the SAME scene with a small, clear pose/expression
# change (a head turn + smile). Per the LTX docs tip, *similar* keyframes
# interpolate best; a small demonstrable delta lets us see if the
# first→last transition actually animates. Generated at the video resolution
# (WIDTH x HEIGHT) so LTX doesn't resample them.
import gc, time, os, subprocess
import torch
from PIL import Image

# Re-run safety: a previously-failed run may have left a partial FLUX pipe in VRAM.
if 'fpipe' in dir():
    del fpipe
    gc.collect()
    torch.cuda.empty_cache()

# ── Park LTX (weights stay on /content; reloaded at the end of this cell) ──
free_ltx()

# ── FLUX.1-dev + character LoRA (same local /content pattern as 01c) ──
# Downloaded straight to /content (local SSD, atomic-safe). NO Drive mirror:
# /content is wiped on a runtime reset, so we re-download here. snapshot_download
# is resumable/idempotent — it skips files already present (a re-run in the same
# session is a no-op) and writes whole files atomically, so no partial/corrupt
# leftovers. Gated -> needs HF token.
FLUX_REPO = 'black-forest-labs/FLUX.1-dev'
FLUX_LOCAL = '/content/flux_dev'
os.makedirs(FLUX_LOCAL, exist_ok=True)

from huggingface_hub import snapshot_download
print(f'Ensuring {FLUX_REPO} on local /content (downloads only what is missing)...')
snapshot_download(FLUX_REPO, local_dir=FLUX_LOCAL, max_workers=8, token=_hf_token() or None)
FLUX_SRC = FLUX_LOCAL
print(f'FLUX source: {FLUX_SRC}\n')

from diffusers import FluxPipeline
fpipe = FluxPipeline.from_pretrained(FLUX_SRC, torch_dtype=torch.bfloat16,
                                     token=os.environ.get('HF_TOKEN') or None)
fpipe.to('cuda')
fpipe.vae.enable_slicing()
fpipe.vae.enable_tiling()

LORA_PATH = f'{DRIVE_BASE}/loras/{CHARACTER_NAME}_flux.safetensors'
assert os.path.exists(LORA_PATH), f'LoRA not found: {LORA_PATH}'
fpipe.load_lora_weights(LORA_PATH)
try:
    fpipe.set_adapters(list(fpipe.get_active_adapters()) or ['default'], [LORA_STRENGTH])   # diffusers >=0.32 two-arg
except Exception:
    fpipe.set_adapters([LORA_STRENGTH])
print(f'FLUX + LoRA @ {LORA_STRENGTH} ready.\n')

def flux_keyframe(prompt, seed, w=WIDTH, h=HEIGHT, steps=28):
    g = torch.Generator(device='cuda').manual_seed(seed)
    return fpipe(prompt=prompt, width=w, height=h, num_inference_steps=steps,
                 guidance_scale=3.5, generator=g, max_sequence_length=512).images[0]

# ── Same scene, small clear delta between the two frames ──────────────────
SCENE = ("in a sunlit café by the window, holding a coffee cup, "
         "warm natural light, shallow depth of field, photorealistic, cinematic")
start_prompt = f'{TRIGGER_TOKEN}, young woman, {SCENE}, ' \
               'looking directly at the camera, relaxed neutral expression'
end_prompt   = f'{TRIGGER_TOKEN}, young woman, {SCENE}, ' \
               'head turned slightly to the side, soft genuine smile'

kf_dir = f'{STILLS_ROOT}/_keyframes'
os.makedirs(kf_dir, exist_ok=True)
ts = time.strftime('%Y%m%d_%H%M%S')

print('Generating START frame...')
start_img = flux_keyframe(start_prompt, seed=7)
START = f'{kf_dir}/{ts}_start.png'
start_img.save(START)
print(f'  -> {START}')
print('Generating END frame...')
end_img = flux_keyframe(end_prompt, seed=8)
END = f'{kf_dir}/{ts}_end.png'
end_img.save(END)
print(f'  -> {END}')

# side-by-side sanity view (start | end)
combo = Image.new('RGB', (start_img.width * 2 + 10, start_img.height), 'black')
combo.paste(start_img, (0, 0)); combo.paste(end_img, (start_img.width + 10, 0))
combo.save(f'{kf_dir}/{ts}_pair.png')
from IPython.display import display
display(combo)
print('\nleft = START (looking at camera)   right = END (turned, smiling)')

# ── Park FLUX, restore LTX ─────────────────────────────────────────────────
del fpipe
gc.collect()
torch.cuda.empty_cache()
print(f'\nFLUX freed — VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB')
pipeline, pipe_upsample = load_ltx()
print(f'READY for §6 — START/END set, LTX reloaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

## 6. Run a clip

In [ ]:
# Motion prompt: describe the ACTION/CAMERA, not the character (frames carry identity).
MOTION_PROMPT = ("She looks up slowly, hair shifting in a light breeze, subtle smile. "
                 "Slow cinematic push-in, shallow depth of field, soft natural light.")

assert START is not None, 'Set START in cell 5 first.'

clip = interpolate(
    start_frame=START,
    end_frame=END,                       # None → image-to-video (start frame only)
    middle_frames=None,                  # e.g. [('mid.png', 0.5)] for a mid keyframe
    prompt=f'{TRIGGER_TOKEN}, {MOTION_PROMPT}',
    seed=0,
    tag='flf2v',
)

from IPython.display import Video, display
display(Video(clip, width=720))

## 7. Chaining clips (longer sequences)
The spec 03 chaining rule: last frame of clip N → start frame of clip N+1. Extract the last frame
with ffmpeg, feed it back into `interpolate()`.

In [ ]:
import subprocess
from pathlib import Path

def last_frame_of(mp4, out_png):
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-sseof', '-0.1', '-i', mp4, '-frames:v', '1', out_png],
                   check=True, capture_output=True)
    return out_png

# Example: extend the clip from §6 by one more segment
# nxt_start = last_frame_of(clip, f'{VID_OUT}/chain/last_frame.png')
# clip2 = interpolate(start_frame=nxt_start, end_frame=END3,
#                     prompt=f'{TRIGGER_TOKEN}, she walks forward through the doorway', tag='seg2')
# Then stitch: ffmpeg concat (see 05_agentic_video.ipynb for the full stitcher)

print('Chaining helpers ready.')

## 8. Commented alternates (bigger/faster/cheaper — try later)
All the other Mode 1 LTX variants we may want, per spec 03's model table.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# A) LTX-Video 2B (base, guidance-distilled 50 steps) — T4 / free tier, ~6-8 GB
#    The main "Lightricks/LTX-Video" repo carries the 2B variant; use the
#    I2V pipeline for single-start-frame generation.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import LTXImageToVideoPipeline
# from diffusers.utils import load_image, export_to_video
# pipe2b = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video",
#     dtype=torch.bfloat16); pipe2b.to('cuda')
# img = load_image(START)
# out = pipe2b(image=img, prompt=MOTION_PROMPT, width=832, height=480, num_frames=161,
#              decode_timestep=0.03, decode_noise_scale=0.025,
#              num_inference_steps=50, guidance_scale=5.0).frames[0]
# export_to_video(out, f'{VID_OUT}/ltx2b.mp4', fps=24)

# ─────────────────────────────────────────────────────────────────────────
# B) GGUF single-file 2B Q3 — runs in ~6 GB (16 GB card / L4 comfortable)
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import LTXPipeline, AutoModel, GGUFQuantizationConfig
# transformer = AutoModel.from_single_file(
#     "https://huggingface.co/city96/LTX-Video-gguf/blob/main/ltx-video-2b-v0.9-Q3_K_S.gguf",
#     quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
#     dtype=torch.bfloat16)
# pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", transformer=transformer,
#                                    dtype=torch.bfloat16)

# ─────────────────────────────────────────────────────────────────────────
# C) 0.9.7-dev + ltxv-spatial-upscaler-0.9.7 (pre-0.9.8 recipe, higher quality
#    ceiling, not distilled → 30 steps + guidance 5.0)
# ─────────────────────────────────────────────────────────────────────────
# pipeline7 = LTXConditionPipeline.from_pretrained("Lightricks/LTX-Video-0.9.7-dev",
#                                                  dtype=torch.bfloat16)
# pipe_up7 = LTXLatentUpsamplePipeline.from_pretrained("Lightricks/ltxv-spatial-upscaler-0.9.7",
#                                                      vae=pipeline7.vae, dtype=torch.bfloat16)
# ... same 4-stage recipe as §4 but timesteps=None, num_inference_steps=30,
#     guidance_scale=5.0, guidance_rescale=0.7

# ─────────────────────────────────────────────────────────────────────────
# D) fp8 layerwise casting for the 13B transformer (saves ~13 GB, tiny speed hit)
#    Use this on a 40GB A100 if group offloading feels slow.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import AutoModel
# transformer = AutoModel.from_pretrained("Lightricks/LTX-Video-0.9.8-13B-distilled",
#     subfolder="transformer", dtype=torch.bfloat16)
# transformer.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn,
#                                      compute_dtype=torch.bfloat16)
# pipeline = LTXConditionPipeline.from_pretrained(MODEL_ID, transformer=transformer,
#                                                 dtype=torch.bfloat16)

# ─────────────────────────────────────────────────────────────────────────
# E) LTX CHARACTER LoRA — once trained (musubi-tuner or an LTX LoRA trainer), load with:
# ─────────────────────────────────────────────────────────────────────────
# pipeline.load_lora_weights("path/or/repo/of/ltx-character-lora", adapter_name="yuna")
# try: pipeline.set_adapters(["yuna"], [1.0])   # diffusers >=0.32 (names, weights)
# except Exception: pipeline.set_adapters([1.0])
# and prefix prompts with the LoRA's trigger word.

print('Section 8: alternates commented out — pick one when needed.')

## 9. Log to metadata

In [ ]:
import json, os, glob, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}

clips = sorted(glob.glob(f'{VID_OUT}/*.mp4'))
meta.setdefault('video_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'mode': 'mode1_ltx',
    'model': 'LTX-Video-0.9.8-13B-distilled',
    'clips': clips[-5:],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — {len(clips)} clips total in {VID_OUT}')
print('\n✅ 03a complete. High-quality FLF2V: 03b (Wan FLF2V). Agentic chaining: 05.')